## Traffic Modeling
#### Starter Pack

In [5]:
import gzip
import struct
import pandas as pd

records = []

#Opens file "TR1.gz" given in assignment PDF
with gzip.open("TR1.gz", "rb") as f:
#with open(file_path, "rb") as f: #if it is autoDecompressed in your MACOS
    while True:
        data = f.read(20)
        if len(data) < 20:
            break

        #Interprets the web traffic that occurred once we took off the quotation marks   
        timestamp, client, obj, size, method, status, file_type, server = \
        struct.unpack(">IIIIBBBB", data)
        records.append([
        timestamp, client, obj, size,
        method, status, file_type, server
        ])

        #Probability and Statistical Modeling of Web Traffic 3

#Puts the interpreted traffic into a readable format
df = pd.DataFrame(records, columns=[
        "timestamp", "client_id", "object_id", "size",
        "method", "status", "type", "server"
        ])
df["datetime"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)
df = df.sort_values("timestamp").reset_index(drop=True)


#Prints the web traffic that occurred
print(df.head())
print(df.shape)


#"Interarrival time between consecutive requests"
df["interarrival_time"] = df["timestamp"].diff()

#"Number of requests per minute"
df["minute"] = df["datetime"].dt.floor("min")
requests_per_minute = df.groupby("minute").size().reset_index(name="request_count")

#Preprocessing is complete above!

   timestamp  client_id  object_id  size  method  status  type  server  \
0  895615201      14275       6219   546       0      66     1      99   
1  895615201       3817        233   631       0      66     1      99   
2  895615201      28686        981  3717       0      66     1      99   
3  895615201     385403        403   993       0      66     1      99   
4  895615201     403925         56  2140       0     130     1      99   

                   datetime  
0 1998-05-19 22:00:01+00:00  
1 1998-05-19 22:00:01+00:00  
2 1998-05-19 22:00:01+00:00  
3 1998-05-19 22:00:01+00:00  
4 1998-05-19 22:00:01+00:00  
(3041727, 9)


### Part 1 - Probability from Observed Events

In [7]:
# A = request belongs to category 1 (type == 1)
# B = object size is exceeds a chosen threshold (greater then the 75th percentile)

A = df["type"] == 1

threshold = df["size"].quantile(0.75)

B = df["size"] > threshold

print("\nThreshold for B:", threshold)

# 1. Estimate P(A), P(B), P(A ∩ B), and P(A ∪ B). Verify the addition rule numerically.
P_A = A.mean()
P_B = B.mean()

# P(A ∩ B)
P_A_and_B = (A & B).mean()
# P(A ∪ B)
P_A_or_B = (A | B).mean()

# Addition rule says that P(A ∪ B) = P(A) + P(B) - P(A ∩ B)
P_A_or_B_addition_rule = (P_A + P_B) - (P_A_and_B) 

print("\n----- Part 1.1 P(A), P(B), P(A ∩ B), P(A ∪ B) and Addition rule -----")
print("P(A):", P_A)
print("P(B):", P_B)
print("P(A ∩ B):", P_A_and_B)
print("P(A ∪ B):", P_A_or_B)
print("Addition rule:", P_A_or_B_addition_rule)

# 2. Estimate P(A | B) and P(B | A). Interpret both probabilities in the context of web requests.
# This are just basically conditional probabilities
P_A_given_B = (P_A_and_B)/(P_B)
P_B_given_A = (P_A_and_B)/(P_A)

print("\n----- Part 1.2 Conditional Probabilities ----- ")
print("P(A | B):", P_A_given_B)
print("P(A | B):", P_B_given_A)

# 3. Investigate whether A and B appear independent. Support your conclusion with numerical evidence.
# We learn in class that Events A and B are independent if any one of the following holds:
# P(A|B) = P(A), P(B|A) = P(B), P(A ∩ B) = P(A)P(B)

P_A_times_P_B = P_A * P_B
print("\n----- Part 1.3 Whether A And B Appear Independent ----- ")
print("Check 1")
print("P(A ∩ B):", P_A_and_B)
print("P(A)P(B):", P_A_times_P_B)
print("Difference:", round(abs(P_A_and_B - P_A_times_P_B),4))

print("\nCheck 2")
print("P(A|B):", P_A_given_B)
print("P(A):", P_A)
print("Difference:", round(abs(P_A_and_B - P_A),4))


print("\nCheck 3")
print("P(B|A):", P_B_given_A)
print("P(B):", P_B)
print("Difference:", round(abs(P_B_given_A - P_B),4))


# 3. Partition requests into several request categories
# Partition requests by type (C) and use it to get P(B)
# P(B) = ∑ P(B|C) * P(C)

P_B_total = 0

print("\n----- 4. Law of total probability ----- ")

for category in df["type"].unique():

    # Here C gets True/False value and get the probability of true
    C = df["type"] == category
    P_C = C.mean()

    # This keeps only the row where C is also true, 
    # we do this because we are looing for the likeliness of a B in given C
    # B has it own True/False list, but we use C as a filter to find type 1 category
    # and calculate the probability from there
    
    P_B_given_C = B[C].mean() 

    contribution = P_B_given_C * P_C
    P_B_total += contribution

    print("\nCategory:", category)
    print("P(C):", P_C)
    print("P(B|C):", P_B_given_C)
    print("P(B|C)P(C):", contribution)

print("\nP(B):", P_B)
print("P(B) using total probability law:", P_B_total)

# 4. Bayes’ theorem to compute the probability 
# Formula: P(C | B) = P(B | C)P(C) / P(B)

selected_category = 1

#This gives C a value of True/ False
C = df["type"] == selected_category

P_C = C.mean()
P_B_given_C = B[C].mean()

P_C_given_B_bayes = (P_B_given_C * P_C) / P_B

# Measured directly
P_C_given_B_direct = (C & B).sum() / B.sum()

print("\n ----- 5. Bayes Theorem -----")
print("Selected category:", selected_category)
print("P(C):", P_C)
print("P(B|C):", P_B_given_C)
print("P(B):", P_B)
print("P(C | B) using Bayes' theorem =", P_C_given_B_bayes)
print("P(C | B) measured directly =", P_C_given_B_direct)
print("Difference =", abs(P_C_given_B_bayes - P_C_given_B_direct))






Threshold for B: 10312.0

----- Part 1.1 P(A), P(B), P(A ∩ B), P(A ∪ B) and Addition rule -----
P(A): 0.8545326388594374
P(B): 0.24995208314224124
P(A ∩ B): 0.19796878549587127
P(A ∪ B): 0.9065159365058074
Addition rule: 0.9065159365058074

----- Part 1.2 Conditional Probabilities ----- 
P(A | B): 0.7920269477538716
P(A | B): 0.2316690744078592

----- Part 1.3 Whether A And B Appear Independent ----- 
Check 1
P(A ∩ B): 0.19796878549587127
P(A)P(B): 0.2135922131959529
Difference: 0.0156

Check 2
P(A|B): 0.7920269477538716
P(A): 0.8545326388594374
Difference: 0.6566

Check 3
P(B|A): 0.2316690744078592
P(B): 0.24995208314224124
Difference: 0.0183

----- 4. Law of total probability ----- 

Category: 1
P(C): 0.8545326388594374
P(B|C): 0.23166907440785917
P(B|C)P(C): 0.19796878549587124

Category: 0
P(C): 0.11748654629425981
P(B|C): 0.38299259574325195
P(B|C)P(C): 0.044996477330148305

Category: 10
P(C): 0.015473446499307795
P(B|C): 0.09119109335826286
P(B|C)P(C): 0.0014110405042924627

Cat